In [0]:
print("Versión de Spark:", spark.version)

Versión de Spark: 4.2.0


In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|actividad11       |
|bd26058           |
|bdexp01           |
|bdexperimento1    |
|default           |
|ef                |
|information_schema|
|n00425513         |
|olist_exp         |
|olist_nuevo       |
|olist_qa          |
|practicaexamen    |
|simulacro         |
|upn26058          |
+------------------+



In [0]:
spark.sql("SHOW TABLES IN workspace.olist_nuevo").show(truncate=False)

+-----------+-----------------+-----------+
|database   |tableName        |isTemporary|
+-----------+-----------------+-----------+
|olist_nuevo|bronze_categorias|false      |
|olist_nuevo|bronze_clientes  |false      |
|olist_nuevo|bronze_detalles  |false      |
|olist_nuevo|bronze_pedidos   |false      |
|olist_nuevo|bronze_productos |false      |
|olist_nuevo|bronze_vendedores|false      |
|olist_nuevo|exp_clientes_x2  |false      |
|olist_nuevo|gold_ventas      |false      |
|olist_nuevo|limpio_categorias|false      |
|olist_nuevo|limpio_clientes  |false      |
|olist_nuevo|limpio_detalles  |false      |
|olist_nuevo|limpio_pedidos   |false      |
|olist_nuevo|limpio_productos |false      |
|olist_nuevo|limpio_vendedores|false      |
|olist_nuevo|silver_categorias|false      |
|olist_nuevo|silver_clientes  |false      |
|olist_nuevo|silver_detalles  |false      |
|olist_nuevo|silver_pedidos   |false      |
|olist_nuevo|silver_productos |false      |
|olist_nuevo|silver_vendedores|f

In [0]:
from pyspark.sql import functions as F

tablas = [
    # BRONZE
    ("Bronze", "bronze_categorias"),
    ("Bronze", "bronze_clientes"),
    ("Bronze", "bronze_detalles"),
    ("Bronze", "bronze_pedidos"),
    ("Bronze", "bronze_productos"),
    ("Bronze", "bronze_vendedores"),

    # SILVER
    ("Silver", "silver_categorias"),
    ("Silver", "silver_clientes"),
    ("Silver", "silver_detalles"),
    ("Silver", "silver_pedidos"),
    ("Silver", "silver_productos"),
    ("Silver", "silver_vendedores"),

    # GOLD
    ("Gold", "gold_ventas")
]

resultados = []

for capa, tabla in tablas:

    nombre_completo = f"workspace.olist_nuevo.{tabla}"
    df = spark.table(nombre_completo)

    # Cantidad de registros
    filas = df.count()

    # Particiones de ejecución que contienen datos
    particiones = (
        df.select(F.spark_partition_id().alias("particion"))
          .distinct()
          .count()
    )

    # Tamaño y metadatos de almacenamiento
    try:
        detalle = spark.sql(
            f"DESCRIBE DETAIL {nombre_completo}"
        ).first().asDict()

        gb = round(detalle["sizeInBytes"] / (1024 ** 3), 6)
        archivos = detalle["numFiles"]
        columnas_particion = str(detalle["partitionColumns"])

    except Exception:
        gb = None
        archivos = None
        columnas_particion = "No disponible"

    resultados.append((
        capa,
        tabla,
        filas,
        gb,
        particiones,
        archivos,
        columnas_particion
    ))

resultado_final = spark.createDataFrame(
    resultados,
    [
        "Capa",
        "Tabla",
        "Filas",
        "GB",
        "Particiones_con_datos",
        "Archivos",
        "Columnas_particionamiento"
    ]
)

display(resultado_final.orderBy("Capa", "Tabla"))

Capa,Tabla,Filas,GB,Particiones_con_datos,Archivos,Columnas_particionamiento
Bronze,bronze_categorias,71,2.0E-6,1,1,[]
Bronze,bronze_clientes,99441,0.003475,1,1,[]
Bronze,bronze_detalles,112650,0.003594,1,1,[]
Bronze,bronze_pedidos,99441,0.005326,1,1,[]
Bronze,bronze_productos,32951,7.4E-4,1,1,[]
Bronze,bronze_vendedores,3096,6.7E-5,1,1,[]
Gold,gold_ventas,108638,0.007567,1,1,[]
Silver,silver_categorias,71,2.0E-6,1,1,[]
Silver,silver_clientes,99441,0.00342,1,1,[]
Silver,silver_detalles,110197,0.003656,1,1,[]
